### Semantic Chunking

In [16]:
sentences = """
    "Configuring iptables rules allows administrators to strictly control inbound network traffic.",
    "Firewalls enforce access control lists to block unauthorized connection attempts.",
    "Systemd service units can be sandboxed using cgroups and network namespace isolation.",
    "Applying SELinux targeted policies prevents compromised daemons from accessing unauthorized file paths.",
    "Port scanning tools like Nmap help security engineers identify exposed network services.",
    "Ronaldo is the goat of football"

    "Photovoltaic solar panels convert sunlight directly into direct current electricity.",
    "Lithium-iron-phosphate batteries are increasingly used for large-scale utility grid storage.",
    "Net metering policies allow residential solar owners to sell excess energy back to the power grid.",
    "Inverters are essential components that convert DC solar output into standard household AC power.",
    "Microgrids improve energy resilience by operating independently during main grid blackouts.",

    "Proper gluten development gives sourdough bread its characteristic open crumb structure.",
    "Proofing dough at lower temperatures overnight enhances flavor development through slow fermentation.",
    "High hydration levels make bread dough sticky and require specialized folding techniques during bulk fermentation.",
    "A preheated Dutch oven traps steam to produce a crispy, blistered crust during baking.",
    "Commercial bakers use baker's percentages to scale ingredient ratios reliably."
"""

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=model_name
)
embeddings

c:\Users\kumar\OneDrive\Desktop\Agentic_AI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2067.63it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [17]:
texts = [sentence.strip() for sentence in sentences.split("\n") if sentence.strip()]

embedds = embeddings.embed_documents(texts)

threshold = 0.5
chunks = []
current_chunk = [texts[0]]

for i in range(1, len(texts)):
    sim = cosine_similarity([embedds[i - 1]], [embedds[i]])[0][0]

    if sim>=threshold:
        current_chunk.append(texts[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [texts[i]]

# append the last chunk
chunks.append(" ".join(current_chunk))

# output the chunks
print("\n Semantic chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\n chunk {idx+1}:\n {chunk}")




 Semantic chunks:

 chunk 1:
 "Configuring iptables rules allows administrators to strictly control inbound network traffic.", "Firewalls enforce access control lists to block unauthorized connection attempts.",

 chunk 2:
 "Systemd service units can be sandboxed using cgroups and network namespace isolation.",

 chunk 3:
 "Applying SELinux targeted policies prevents compromised daemons from accessing unauthorized file paths.",

 chunk 4:
 "Port scanning tools like Nmap help security engineers identify exposed network services.",

 chunk 5:
 "Ronaldo is the goat of football"

 chunk 6:
 "Photovoltaic solar panels convert sunlight directly into direct current electricity.",

 chunk 7:
 "Lithium-iron-phosphate batteries are increasingly used for large-scale utility grid storage.",

 chunk 8:
 "Net metering policies allow residential solar owners to sell excess energy back to the power grid.", "Inverters are essential components that convert DC solar output into standard household AC pow

### RAG pipeline with modular coding

In [21]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser ## extract output that parses LLMresult into a top likely string
from langchain_core.runnables import RunnablePassthrough, RunnableMap
from langchain_core.prompts import PromptTemplate
import os

# 🤖 LangChain: `init_chat_model` Cheat Sheet

The `init_chat_model()` function provides a single initializer for all chat model integrations in LangChain, replacing provider-specific imports.

---

## 🔑 Key Benefits
* **Unified API:** Use one function to load models across providers (OpenAI, Anthropic, Mistral, Google, local Ollama, etc.).
* **Model Agnostic Pipelines:** Swap model backends without changing downstream chain or agent logic.
* **Configurable Models:** Infer credentials and model parameters dynamically at runtime.

---

## 📦 Basic Usage Example

```python
from langchain.chat_models import init_chat_model

# Initialize OpenAI GPT-4o
gpt_model = init_chat_model("gpt-4o", model_provider="openai", temperature=0)

# Initialize Anthropic Claude 3.5 Sonnet
claude_model = init_chat_model("claude-3-5-sonnet-20240620", model_provider="anthropic", temperature=0)

# Initialize local Ollama model
local_model = init_chat_model("llama3", model_provider="ollama")

# Invoke standard LCEL interface
response = gpt_model.invoke("Explain zero-trust architecture in one sentence.")
print(response.content)

In [ ]:
os.environ["GROQ_API_KEY"] = groq_api_key

groq_llm = init_chat_model(model="groq:llama-3.3-70b-versatile")

groq_llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021AA8696990>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021AA989DEB0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [41]:
### Custom semantic chunking with variable threshold

class SemanticChunking:
    def __init__(self,model_name="all-MiniLM-L6-v2", threshold=0.5):
        self.model = HuggingFaceEmbeddings(model_name=model_name)
        self.threshold = threshold

    def splitDocs(self, sentences: str):
        sentences = [sentence.strip() for sentence in sentences.split("\n") if sentence.strip()]
        embedds = self.model.embed_documents(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embedds[i - 1]], [embedds[i]])[0][0]
            if sim >= threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + '.')
                current_chunk = [sentences[i]]
        chunks.append(". ".join(current_chunk) + '.')
        return chunks

    def split_documents(self, docs):
        documents = []
        for chunk in self.splitDocs(docs.page_content):
            documents.append(Document(page_content=chunk, metadata=doc.metadata))
        return documents

In [42]:
doc = Document(page_content=sentences)

doc

Document(metadata={}, page_content='\n    "Configuring iptables rules allows administrators to strictly control inbound network traffic.",\n    "Firewalls enforce access control lists to block unauthorized connection attempts.",\n    "Systemd service units can be sandboxed using cgroups and network namespace isolation.",\n    "Applying SELinux targeted policies prevents compromised daemons from accessing unauthorized file paths.",\n    "Port scanning tools like Nmap help security engineers identify exposed network services.",\n    "Ronaldo is the goat of football"\n\n    "Photovoltaic solar panels convert sunlight directly into direct current electricity.",\n    "Lithium-iron-phosphate batteries are increasingly used for large-scale utility grid storage.",\n    "Net metering policies allow residential solar owners to sell excess energy back to the power grid.",\n    "Inverters are essential components that convert DC solar output into standard household AC power.",\n    "Microgrids imp

In [43]:
chunker = SemanticChunking(threshold=0.5)

chunks = chunker.split_documents(doc)

chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3727.24it/s]


[Document(metadata={}, page_content='"Configuring iptables rules allows administrators to strictly control inbound network traffic.",. "Firewalls enforce access control lists to block unauthorized connection attempts.",.'),
 Document(metadata={}, page_content='"Systemd service units can be sandboxed using cgroups and network namespace isolation.",.'),
 Document(metadata={}, page_content='"Applying SELinux targeted policies prevents compromised daemons from accessing unauthorized file paths.",.'),
 Document(metadata={}, page_content='"Port scanning tools like Nmap help security engineers identify exposed network services.",.'),
 Document(metadata={}, page_content='"Ronaldo is the goat of football".'),
 Document(metadata={}, page_content='"Photovoltaic solar panels convert sunlight directly into direct current electricity.",.'),
 Document(metadata={}, page_content='"Lithium-iron-phosphate batteries are increasingly used for large-scale utility grid storage.",.'),
 Document(metadata={}, p

In [44]:
vector_store = FAISS.from_documents(chunks, embedding=embeddings)

retriever = vector_store.as_retriever()

In [45]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000021AA893BA40>, search_kwargs={})

In [46]:
## Prompt template

template = """
Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template=template)

prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [67]:
## LLM model

os.environ["GROQ_API_KEY"] = groq_api_key

groq_llm = init_chat_model(model="openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0.2)

groq_llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021AB3FBE000>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021AB6811F40>, model_name='openai/gpt-oss-120b', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

LCEL is a declarative syntax for stitching together LLM components into production-ready pipelines.

---

## 🎯 What is LCEL Used For?

LCEL provides a unified interface (`invoke`, `stream`, `batch`, `ainvoke`) across components like LLMs, Prompt Templates, Vector Stores, and Output Parsers.

* **Parallel Execution:** Automatically executes independent steps concurrently to minimize latency.
* **Streaming Support:** Stream tokens natively from the model to the client UI.
* **Built-in Async:** Works seamlessly in async Python environments (`asyncio`).
* **Easy Fallbacks & Retries:** Configure model or API fallbacks directly on any runnable step.

---

## 🗺️ What is `RunnableMap`?

`RunnableMap` runs multiple operations in parallel and returns their results formatted as a Python dictionary. 

In modern LCEL, `RunnableMap` is implicitly created whenever you pass a standard dictionary `{}` into a pipe chain (`|`).

### Core Use Cases:
1. **Parallel Data Retrieval:** Fetch vector embeddings, SQL context, and user metadata simultaneously.
2. **Prompt Structuring:** Format multiple dynamic inputs into the keys required by a `PromptTemplate`.
3. **Branching Pipelines:** Process a single input through different prompts or models concurrently.

---

## 🆚 Pipe Operator (`|`) vs `RunnableMap`

* **Pipe Operator (`|`):** Connects runnables sequentially. Output of the left side becomes input to the right side:
  $$\text{Input} \longrightarrow \text{Step 1} \longrightarrow \text{Step 2} \longrightarrow \text{Output}$$
* **`RunnableMap` (or `{}`):** Executes multiple runnables concurrently on the exact same input and returns a structured dictionary:
  $$\text{Input} \longrightarrow \begin{cases} \text{Branch A} \longrightarrow \text{dict["key\_a"]} \\ \text{Branch B} \longrightarrow \text{dict["key\_b"]} \end{cases}$$

---

In [63]:
### LCEL

rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"]
        }
    )
    | prompt
    | groq_llm
    | StrOutputParser()
)

query = {"question": "What is SELinux?"}
result = rag_chain.invoke(query)

In [64]:
result

'SELinux (Security‑Enhanced Linux) is a Linux security framework that implements mandatory access‑control policies. By applying targeted policies, it restricts processes—such as daemons—from accessing file paths and other system resources they are not authorized to use, thereby helping to contain compromised services.'

### Semantic Chunking with langchain (in-built function)

In [66]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader